In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
import pandas as pd

df = pd.read_csv("trimmed-data-nlp.csv")

# combine text columns first
text_cols = ["Significance Summary", "Description", "Heritage Value", "Character Defining Elements"]
df["combined_text"] = df[text_cols].fillna("").apply(lambda row: " ".join(row), axis=1)

# encode text
st_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = st_model.encode(df["combined_text"].tolist())
embedding_df = pd.DataFrame(embeddings, index=df.index)

# build features
X = df.drop(columns=["Municipal", "Is Demolished", "combined_text"] + text_cols)
y = df["Municipal"]
X = pd.get_dummies(X)

# add embeddings
X = pd.concat([X, embedding_df], axis=1)

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train
model = XGBClassifier(random_state=42, scale_pos_weight=807/175)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

c:\Users\canmo\Desktop\Winter 2026\DATA 3471\COMP3471-HeritageCalgary-WIL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3909.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


              precision    recall  f1-score   support

       False       0.85      0.97      0.91       160
        True       0.69      0.24      0.36        37

    accuracy                           0.84       197
   macro avg       0.77      0.61      0.63       197
weighted avg       0.82      0.84      0.80       197



In [2]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_proba >= 0.4).astype(bool)
print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

       False       0.85      0.96      0.90       160
        True       0.60      0.24      0.35        37

    accuracy                           0.83       197
   macro avg       0.72      0.60      0.62       197
weighted avg       0.80      0.83      0.80       197



In [3]:
y_train_pred = model.predict(X_train)
print("Train:")
print(classification_report(y_train, y_train_pred))
print("Test:")
print(classification_report(y_test, y_pred))

Train:
              precision    recall  f1-score   support

       False       1.00      1.00      1.00       647
        True       0.99      1.00      1.00       138

    accuracy                           1.00       785
   macro avg       1.00      1.00      1.00       785
weighted avg       1.00      1.00      1.00       785

Test:
              precision    recall  f1-score   support

       False       0.85      0.97      0.91       160
        True       0.69      0.24      0.36        37

    accuracy                           0.84       197
   macro avg       0.77      0.61      0.63       197
weighted avg       0.82      0.84      0.80       197



In [4]:
undesignated = df[df["Municipal"] == False].copy()

# prepare features same way
undesignated["combined_text"] = undesignated[text_cols].fillna("").apply(lambda row: " ".join(row), axis=1)
embeddings_new = st_model.encode(undesignated["combined_text"].tolist())
embedding_df_new = pd.DataFrame(embeddings_new, index=undesignated.index)

X_undesignated = undesignated.drop(columns=["Municipal", "Is Demolished", "combined_text"] + text_cols)
X_undesignated = pd.get_dummies(X_undesignated)
X_undesignated = pd.concat([X_undesignated, embedding_df_new], axis=1)
X_undesignated = X_undesignated.reindex(columns=X_train.columns, fill_value=0)

undesignated["probability"] = model.predict_proba(X_undesignated)[:, 1]
results = undesignated[["Name", "Community", "Year of Construction", "probability"]].sort_values("probability", ascending=False)
print(results.head(20))

results.to_csv("results-nlp.csv")

                                                  Name  \
100                                     Betz Residence   
266                                      Crescent Park   
346                             Fire Hall No. 5 (1952)   
254                                 Costigan Residence   
120                                  Bowness Town Hall   
834                         St. Andrew's United Church   
929                          W.H. Birkenshaw Residence   
350  Firestone Tire & Rubber Company of Canada (wat...   
215                      Central Memorial Park Library   
451                                   Hillhurst School   
732                                     Reid Residence   
893                                   Teskey Residence   
49                         Alberta Wheat Pool Building   
750                    Riverside Bungalow School No. 2   
863                        St. Stephen's Memorial Hall   
861  St. Stephen’s Anglican Church and Education Bu...   
213           

In [6]:
# check column names first
print(results_basic.columns.tolist())
print(results_nlp.columns.tolist())

['Unnamed: 0', 'Name', 'Community', 'Year of Construction', 'probability', 'prob_basic']
['Unnamed: 0', 'Name', 'Community', 'Year of Construction', 'probability', 'prob_nlp']


In [ ]:
results_basic = pd.read_csv("results-basic.csv")
results_nlp = pd.read_csv("results-nlp.csv")

merged = results_basic[["Name", "probability"]].rename(columns={"probability": "prob_basic"}).merge(
    results_nlp[["Name", "probability"]].rename(columns={"probability": "prob_nlp"}),
    on="Name"
)
merged["difference"] = abs(merged["prob_basic"] - merged["prob_nlp"])
merged.sort_values("difference", ascending=False).head(20)



,Name,prob_basic,prob_nlp,difference
0,Court House No. 2,0.953200,0.001243,0.951957
2,7 Street NW Boulevards,0.903557,0.047007,0.856550
3,Hotel Cecil (Cecil Hotel),0.844308,0.004429,0.839879
4,Bow Valley Lawn Bowling Club,0.837863,0.004189,0.833675
5,Substation No. Four,0.821506,0.002669,0.818837
7,Louise (Hillhurst) Bridge,0.810332,0.006147,0.804185
8,Rotary Park Lawn Bowls,0.804588,0.001134,0.803454
11,Allen (Palace) Theatre,0.786457,0.001236,0.785221
6,Anderson Apartments,0.812905,0.042096,0.770809
9,Fort Calgary Archaeological Site,0.798854,0.031161,0.767693


In [12]:
merged["difference"] = abs(merged["prob_nlp"] - merged["prob_basic"])
merged.sort_values("difference", ascending=False).head(20)

,Name,prob_basic,prob_nlp,difference
0,Court House No. 2,0.953200,0.001243,0.951957
2,7 Street NW Boulevards,0.903557,0.047007,0.856550
3,Hotel Cecil (Cecil Hotel),0.844308,0.004429,0.839879
4,Bow Valley Lawn Bowling Club,0.837863,0.004189,0.833675
5,Substation No. Four,0.821506,0.002669,0.818837
7,Louise (Hillhurst) Bridge,0.810332,0.006147,0.804185
8,Rotary Park Lawn Bowls,0.804588,0.001134,0.803454
11,Allen (Palace) Theatre,0.786457,0.001236,0.785221
6,Anderson Apartments,0.812905,0.042096,0.770809
9,Fort Calgary Archaeological Site,0.798854,0.031161,0.767693


In [11]:
print(results_nlp.head(20))

    Unnamed: 0                                               Name  \
0          100                                     Betz Residence   
1          266                                      Crescent Park   
2          346                             Fire Hall No. 5 (1952)   
3          254                                 Costigan Residence   
4          120                                  Bowness Town Hall   
5          834                         St. Andrew's United Church   
6          929                          W.H. Birkenshaw Residence   
7          350  Firestone Tire & Rubber Company of Canada (wat...   
8          215                      Central Memorial Park Library   
9          451                                   Hillhurst School   
10         732                                     Reid Residence   
11         893                                   Teskey Residence   
12          49                        Alberta Wheat Pool Building   
13         750                    